# Step 5b — DistilBERT cross-encoder

Step 4's ESIM with DistilBERT as the encoder: one joined row `[CLS] A [SEP] B [SEP]` (A = Overture first), masked mean + max pooling over the row, MLP 200 (tanh) → 1 logit.

Spec: `step5_spec.md`.

In [34]:
# Colab only (does nothing locally): mount Google Drive and work from the project's copy there.
# After this, every relative path (sys.path "..", load_pairs, CHECKPOINT_DIR = "checkpoints") points into Drive,
# so checkpoints survive a disconnect.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/geocoding_ml/transformers")
    print("Colab: working in", os.getcwd())
except ImportError:
    pass

## 1. Setup

In [47]:
import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from helpers.pairs import load_pairs
from siamese_model.data_plumbing import PairDataset
from torch.utils.data import DataLoader, Dataset, Subset
from torch import nn
import os, copy
from helpers.check_model_performance import review_typical_slices   # slice table used by evaluate
# only these four: real_training's own train_epoch / predict are Step 3's CPU versions
from siamese_model.real_training import is_better, broken_run_problems, THRESHOLD, CASE1_GUARDRAIL


DATASET = "overture_case2s"
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available()          # Colab GPU
                      else "mps" if torch.backends.mps.is_available()  # M1 Pro
                      else "cpu")
BATCH_SIZE = 32
MLP_HIDDEN, DROPOUT = 200, 0.3   # head, as in ESIM
MAX_EPOCHS = 5
PATIENCE = 2
SEEDS = [0, 1, 2]
CHECKPOINT_DIR = "checkpoints_cross"   # git-ignored; its own folder so 5a's seed files are never skipped or resumed

train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
train.shape, val.shape, test.shape, DEVICE

((202568, 8), (41865, 8), (43844, 8), device(type='mps'))

## 2. Tokenise + collate

Pairs tokenised together: `tok(address_a, address_b, ...)`, `padding=True`, `max_length=MAX_LENGTH`.

In [28]:
def make_collate():
    def collate(samples):
        addr_a, addr_b, labels = zip(*samples)
        lengths = torch.tensor([len(x) for x in tok(addr_a, addr_b)["input_ids"]])

        ids = tok(addr_a, addr_b, padding=True, return_tensors="pt")
        mask = (
            torch.arange(lengths.max())[None, :] < lengths[:, None]).long()
        
        assert torch.equal(mask, ids["attention_mask"])
        y = torch.tensor(labels, dtype=torch.float32)
        return ids["input_ids"], mask, y 
    return collate


train_dataset = PairDataset(train, train_label)
val_dataset = PairDataset(val, val_label)
test_dataset = PairDataset(test, test_label)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle = True, collate_fn = make_collate())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle = False, collate_fn= make_collate())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle = False, collate_fn= make_collate())

In [31]:
ids, mask, y = next(iter(train_loader))
assert ((ids == tok.sep_token_id).sum(1) == 2).all()
assert tok.decode(ids[0]).startswith("[CLS]")
assert (mask.sum(1) == (ids != tok.pad_token_id).sum(1)).all()

## 3. Encoder: DistilBERT + masked mean + max pooling

Tests directly under the class.

In [36]:
class MeanMaxPooling(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, h, mask):
        m = mask.unsqueeze(-1).bool()  # (B, T, 1), True = real token; broadcasts over features
        mean_pool = (h * m).sum(1) / m.sum(1) # pads add 0 and don't count: (B, T, D) -> (B, D)
        max_pool = h.masked_fill(~m, float("-inf")).max(dim=1).values
        return torch.cat([mean_pool, max_pool], dim = 1)


class BertEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = MeanMaxPooling()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.output_dim = 2 * self.bert.config.hidden_size

    def forward(self,ids, mask):
        bert_res = self.bert(input_ids=ids, attention_mask=mask)
        return self.pool(bert_res.last_hidden_state, mask)

In [37]:
h = torch.tensor([[[1., 1.], [9., 9.]]])   # 1 row, 2 positions; position 2 is a pad
mask = torch.tensor([[1, 0]])
assert torch.equal(MeanMaxPooling()(h, mask), torch.tensor([[1., 1., 1., 1.]]))

## 4. Cross-encoder: head + forward

MLP 200, tanh, dropout per spec. Tests directly under the class.

In [41]:
class ClassifierHead(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, MLP_HIDDEN) 
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(DROPOUT)
        self.linear2 = nn.Linear(MLP_HIDDEN, 1)
        

    def forward(self, vectors):
        linear1_val = self.linear1(vectors)
        tanh_val = self.tanh(linear1_val)
        dropout_val = self.dropout(tanh_val)
        linear2_val = self.linear2(dropout_val)
        return linear2_val.squeeze(-1)

# ESIM model
class CrossEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = BertEncoder()
        self.head = ClassifierHead(self.encoder.output_dim)

    def forward(self, ids, mask):
        res = self.encoder(ids, mask)    
        return self.head(res)

## 5. Optimiser + scheduler

AdamW, two groups: encoder 2e-5, head 1e-3; weight decay 0.01; 10% linear warm-up.

In [42]:
model = CrossEncoder()
N_EPOCHS = 10
optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 2e-5},
    {"params": model.head.parameters(),    "lr": 1e-3},
], weight_decay=0.01)

total_steps = len(train_loader) * N_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

## 6. Timing: one epoch on a 10k train slice

Batch 32, on `DEVICE`. Decides epoch budget and M1 Pro vs Colab.

In [44]:
def to_device(batch):
    return (t.to(DEVICE) for t in batch)                             # all three tensors: ids, mask, y

def sync():
    # GPUs run asynchronously: wait until the queued work is done before reading the clock
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()

def train_epoch(model, loader, opt, sched, loss_fn):
    model.train()
    total_loss, n_pairs = 0.0, 0
    for batch in loader:
        ids, mask, y = to_device(batch)
        opt.zero_grad()                                              # first: clear anything left from before
        logits = model(ids, mask)                                    # the model never sees y
        loss = loss_fn(logits, y)                                    # BCEWithLogitsLoss applies the sigmoid itself
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # between backward and step: edits the gradients step uses
        opt.step()
        sched.step()                                                 # after opt.step(): sets the lr for the next batch
        total_loss += loss.item() * len(y)                           # weight by batch size -> average per pair, not per batch
        n_pairs += len(y)
    return total_loss / n_pairs

def predict(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():                                            # no graph recorded, no updates
        for batch in loader:
            ids, mask, y = to_device(batch)                          # y unused
            logits = model(ids, mask)
            probs.append(torch.sigmoid(logits).cpu())                # logits -> probabilities, back on the CPU
    return torch.cat(probs)                                          # (N,), in loader order: val/test loaders must not shuffle

In [46]:
import time
SEED = 0
N_TIMING_BATCHES = 200

torch.manual_seed(SEED)
timing_model = CrossEncoder().to(DEVICE)
timing_opt = torch.optim.AdamW([
    {"params": timing_model.encoder.parameters(), "lr": 2e-5},
    {"params": timing_model.head.parameters(),    "lr": 1e-3},
], weight_decay=0.01)
timing_steps = N_TIMING_BATCHES  # one pass over the timing slice
timing_sched = get_linear_schedule_with_warmup(
    timing_opt, num_warmup_steps=int(0.1 * timing_steps), num_training_steps=timing_steps)

# random rows: the first train rows are short identical positives and would understate the time
rng = np.random.default_rng(SEED)
timing_idx = rng.choice(len(train_dataset), N_TIMING_BATCHES * BATCH_SIZE, replace=False)
subset_loader = DataLoader(Subset(train_dataset, timing_idx), batch_size=BATCH_SIZE, collate_fn=make_collate())

# train/val loaders come from §2 (train shuffled); don't rebuild them here

t0 = time.perf_counter()
train_epoch(timing_model, subset_loader, timing_opt, timing_sched, nn.BCEWithLogitsLoss())
sync()
train_seconds = time.perf_counter() - t0
print(f"train: {train_seconds / N_TIMING_BATCHES:.3f} s/batch -> {train_seconds / N_TIMING_BATCHES * len(train_loader) / 60:.1f} min/epoch")

t0 = time.perf_counter()
predict(timing_model, val_loader)
sync()
print(f"val pass: {time.perf_counter() - t0:.1f} s")

del timing_model, timing_opt, timing_sched
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
elif DEVICE.type == "mps":
    torch.mps.empty_cache()

train: 0.153 s/batch -> 16.1 min/epoch
val pass: 35.3 s


## 7. Training

Best epoch = highest val non-identical found with val case1 ≥ 98%.

In [48]:
def evaluate(model, loader, df, labels):
    """Copied from siamese_model/real_training.py; calls this notebook's device-aware predict."""
    probs = predict(model, loader)
    y_true = torch.tensor(labels.values, dtype=torch.float32)
    preds = (probs >= THRESHOLD).int().numpy()
    val_loss = nn.functional.binary_cross_entropy(probs, y_true).item()
    table = review_typical_slices(df, preds, labels).set_index("slice")
    pct = lambda name: float(100 * table.loc[name, "correct"] / table.loc[name, "n"])
    found = pct("positives: non-identical")
    case1 = pct("neg: case1 (same street, diff number)")
    easy = pct("neg: case3 (easy)")
    return {"val_loss": val_loss, "found": found, "case1": case1, "easy": easy}


def run_training(seed, train_loader=train_loader, checkpoint_dir=CHECKPOINT_DIR, max_epochs=MAX_EPOCHS):
    # train_loader / checkpoint_dir / max_epochs are arguments so the dry run can't leave a global changed
    torch.manual_seed(seed)
    last_path = f"{checkpoint_dir}/seed_{seed}_last.pt"
    final_path = f"{checkpoint_dir}/seed_{seed}_final.pt"

    model = CrossEncoder().to(DEVICE)
    model.train()
    opt = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": 2e-5},
        {"params": model.head.parameters(),    "lr": 1e-3},
    ], weight_decay=0.01)
    total_steps = len(train_loader) * max_epochs
    sched = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    loss_fn = nn.BCEWithLogitsLoss()

    best, best_state, history = None, None, []
    epochs_since_best, start_epoch = 0, 0
    
    if os.path.exists(last_path):                      # a disconnected run: continue after its last finished epoch
        ck = torch.load(last_path, map_location=DEVICE)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        history, best, best_state, epochs_since_best = ck["history"], ck["best"], ck["best_state"], ck["epochs_since_best"]
        start_epoch = ck["epoch"] + 1
        print(f"seed {seed}: resuming at epoch {start_epoch}")
    
    for epoch in range(start_epoch, max_epochs):
        train_loss = train_epoch(model, train_loader, opt,sched, loss_fn)
        
        row = {"epoch": epoch, "train_loss": train_loss,"lr_encoder": sched.get_last_lr()[0], **evaluate(model, val_loader, val, val_label)}
        print(row)
        history.append(row)

        

        if is_better(row, best):
            best, best_state, epochs_since_best = row, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_since_best += 1
            if epochs_since_best >= PATIENCE:
                break

        state = {
            "epoch": epoch, "model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), 
            "history": history, "best": best,
            "best_state": best_state, "epochs_since_best": epochs_since_best,
        }
        torch.save(state, last_path + ".tmp")
        os.replace(last_path + ".tmp", last_path)
        
    # no epoch reached the case1 guardrail -> keep the last epoch's weights
    if best is None:
        best_state = copy.deepcopy(model.state_dict())
    
    torch.save({"seed": seed, "best_state": best_state, "best": best, "history": history}, final_path)
    
    if os.path.exists(last_path):
        os.remove(last_path)

    return pd.DataFrame(history), best, best_state

In [50]:
import shutil
dry_loader = DataLoader(Subset(train_dataset, range(64)), batch_size = 8, shuffle = True, collate_fn = make_collate())
os.makedirs("checkpoints_dry", exist_ok = True)   # git-ignored, in case the run crashes before the cleanup
dry_history, dry_best, dry_best_state= run_training(0, dry_loader, "checkpoints_dry", max_epochs = 2)
display(dry_history)   # not the last line -> needs display()

shutil.rmtree("checkpoints_dry")   # os.remove only deletes files, not folders


{'epoch': 0, 'train_loss': 0.2060471724398667, 'lr_encoder': 1.0666666666666667e-05, 'val_loss': 5.959127902984619, 'found': 100.0, 'case1': 0.0, 'easy': 0.0}
{'epoch': 1, 'train_loss': 0.001106647338019684, 'lr_encoder': 0.0, 'val_loss': 6.191230297088623, 'found': 100.0, 'case1': 0.0, 'easy': 0.0}


,epoch,train_loss,lr_encoder,val_loss,found,case1,easy
0,0,0.206047,0.000011,5.959128,100.0,0.0,0.0
1,1,0.001107,0.000000,6.191230,100.0,0.0,0.0


In [ ]:

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
for seed in SEEDS:
    if os.path.exists(f"{CHECKPOINT_DIR}/seed_{seed}_final.pt"):
        print(f"seed {seed}: already finished, skipping")
        continue
    history, best, best_state = run_training(seed)
    print(seed, broken_run_problems(history))

{'epoch': 0, 'train_loss': 0.055513869508720365, 'lr_encoder': 1.777746577746578e-05, 'val_loss': 0.032574720680713654, 'found': 71.14178168130489, 'case1': 99.93469269283797, 'easy': 99.99564630589055}
{'epoch': 1, 'train_loss': 0.028676954462670698, 'lr_encoder': 1.3333099333099335e-05, 'val_loss': 0.026632575318217278, 'found': 63.237139272271015, 'case1': 99.96371816268775, 'easy': 100.0}
{'epoch': 2, 'train_loss': 0.02365139311037619, 'lr_encoder': 8.88873288873289e-06, 'val_loss': 0.032066710293293, 'found': 73.27478042659975, 'case1': 99.89115448806328, 'easy': 99.98693891767165}
{'epoch': 3, 'train_loss': 0.01929914065255638, 'lr_encoder': 4.444366444366445e-06, 'val_loss': 0.029790982604026794, 'found': 74.27854454203262, 'case1': 99.94194906030042, 'easy': 99.99564630589055}
{'epoch': 4, 'train_loss': 0.015735004941846993, 'lr_encoder': 0.0, 'val_loss': 0.02960079535841942, 'found': 77.66624843161857, 'case1': 99.89115448806328, 'easy': 99.98693891767165}
0 []
{'epoch': 0, 't

## 8. Results

Slice table on val, then test; rare-number case1 group. Swap test deferred.

## Diagnostics

One-off queries and their results (question → result → query).